# Proyek Klasifikasi Gambar: Fast Food Classification (v2 - Fine-Tuned + OOD Gate)

### Ringkasan Peningkatan v2
1. **Pipeline `tf.data` Cepat + `uint8` RAM Cache:** Mengganti `ImageDataGenerator` (single-threaded CPU) dengan `image_dataset_from_directory(crop_to_aspect_ratio=True)`, `.cache()` dalam `uint8` (~1,6 GB RAM), augmentasi native `tf.image` paralel, `.prefetch(AUTOTUNE)`, dan `BATCH_SIZE = 64` sehingga pelatihan 2-stage di Colab T4 GPU selesai dalam ~3-5 menit.
2. **Perbaikan Normalisasi MobileNetV2:** Menggunakan skala `[-1, 1]` (`(x / 127.5) - 1.0`) sesuai bobot pra-latih ImageNet.
3. **2-Stage Fine-Tuning & Checkpoint Terbaik:** Stage 1 melatih head (`Conv2D + BatchNormalization + ReLU + MaxPooling2D + GAP + Dense + Dropout`), diikuti Stage 2 membuka 40 layer atas MobileNetV2 (`lr=3e-5`, `BatchNormalization` backbone tetap freeze) dengan `ModelCheckpoint(monitor='val_loss')` dan `label_smoothing=0.1`.
4. **Kelas ke-7 (`Other`) + OOD Embedding Gate:** Menggabungkan 4 kelas sisa Fast Food V2 (`Hot Dog, Sandwich, Taco, Taquito`) dan sampel *hard-negative* non-fast-food dari Food-11 (`Seafood, Meat, Rice, Soup, Noodles-Pasta, Vegetable-Fruit`) sebagai kelas `Other`, serta menghitung *centroid* fitur 128-dim pada layer `embedding_gap` untuk `models/model_meta.json`.
5. **Ekspor Browser-Ready:** Otomatis mengekspor `saved_model/`, `tflite/`, `tfjs_model/` (skema `model_config` terverifikasi untuk TensorFlow.js), dan `model_meta.json` ke arsip `models_v2.zip`.

## 1. Setup Lingkungan & Import Library

In [ ]:
import os
# Gunakan legacy Keras (Keras 2 schema) di Colab agar ekspor TFJS layers-model kompatibel penuh di browser.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip install -q kagglehub scikit-learn

import json
import random
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import kagglehub
from PIL import Image
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", gpus)
if not gpus:
    print("[PERINGATAN] GPU tidak terdeteksi! Aktifkan Runtime > Change runtime type > T4 GPU agar training cepat.")

## 2. Data Loading, Perakitan Kelas `Other` (Hard Negatives), & Audit Resolusi

In [ ]:
FAST_FOOD_CLASSES = ("Baked Potato", "Burger", "Crispy Chicken", "Donut", "Fries", "Pizza")
UNUSED_FAST_FOOD_CLASSES = ("Hot Dog", "Sandwich", "Taco", "Taquito")
FOOD11_OTHER_QUOTAS = {
    "Seafood": 400,
    "Meat": 200,
    "Rice": 200,
    "Soup": 150,
    "Noodles-Pasta": 150,
    "Vegetable-Fruit": 100,
}
OTHER_CLASS_NAME = "Other"
SOURCE_SPLITS = ("Train", "Valid", "Test")
SEED = 42

ff_path = Path(kagglehub.dataset_download("utkarshsaxenadn/fast-food-classification-dataset"))
source_base = ff_path / "Fast Food Classification V2"
print("Fast Food dataset:", source_base)

food11_root = Path(kagglehub.dataset_download("trolukovich/food11-image-dataset"))
print("Food-11 dataset:", food11_root)

raw_dir = Path("/content/raw_dataset_v2")
if raw_dir.exists():
    shutil.rmtree(raw_dir)
raw_dir.mkdir(parents=True, exist_ok=True)


def is_valid_image(path: Path) -> bool:
    if not path.is_file():
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def gather_by_folder_name(root: Path, folder_name: str) -> list[Path]:
    target = folder_name.lower()
    files = []
    for sub in sorted(root.rglob("*")):
        if sub.is_dir() and sub.name.lower() == target:
            files.extend(sorted(p for p in sub.iterdir() if p.is_file()))
    return files


for cls in FAST_FOOD_CLASSES:
    cls_dir = raw_dir / cls
    cls_dir.mkdir(parents=True, exist_ok=True)
    counter = 0
    for sp in SOURCE_SPLITS:
        src_dir = source_base / sp / cls
        if not src_dir.is_dir():
            continue
        for src_file in sorted(src_dir.iterdir()):
            if not is_valid_image(src_file):
                continue
            shutil.copy2(src_file, cls_dir / f"{sp}_{counter}{src_file.suffix}")
            counter += 1

other_dir = raw_dir / OTHER_CLASS_NAME
other_dir.mkdir(parents=True, exist_ok=True)
rng = random.Random(SEED)
other_counter = 0

for cls in UNUSED_FAST_FOOD_CLASSES:
    candidates = []
    for sp in SOURCE_SPLITS:
        src_dir = source_base / sp / cls
        if src_dir.is_dir():
            candidates.extend(sorted(p for p in src_dir.iterdir() if p.is_file()))
    rng.shuffle(candidates)
    copied = 0
    for src_file in candidates:
        if copied >= 300:
            break
        if not is_valid_image(src_file):
            continue
        slug = cls.lower().replace(" ", "_")
        shutil.copy2(src_file, other_dir / f"ff_{slug}_{other_counter}{src_file.suffix}")
        copied += 1
        other_counter += 1

for category, quota in FOOD11_OTHER_QUOTAS.items():
    candidates = gather_by_folder_name(food11_root, category)
    rng.shuffle(candidates)
    copied = 0
    for src_file in candidates:
        if copied >= quota:
            break
        if not is_valid_image(src_file):
            continue
        slug = category.lower().replace("-", "_").replace(" ", "_")
        shutil.copy2(src_file, other_dir / f"f11_{slug}_{other_counter}{src_file.suffix}")
        copied += 1
        other_counter += 1


def inspect_resolutions(directory: Path) -> None:
    overall_sizes = set()
    total = 0
    for subdir in sorted(p for p in directory.iterdir() if p.is_dir()):
        files = [p for p in subdir.iterdir() if p.is_file()]
        cls_sizes = set()
        for f in files:
            with Image.open(f) as im:
                overall_sizes.add(im.size)
                cls_sizes.add(im.size)
        total += len(files)
        print(f"{subdir.name:<16}: {len(files):4d} gambar | {len(cls_sizes):3d} variasi resolusi")
    print(f"\nTotal gambar: {total} | Total resolusi unik: {len(overall_sizes)}")


inspect_resolutions(raw_dir)

## 3. Stratified Split (80:10:10) & Pipeline `tf.data` Cepat (`uint8` RAM Cache + `[-1, 1]`)

In [ ]:
split_dir = Path("/content/dataset_split_v2")
if split_dir.exists():
    shutil.rmtree(split_dir)

train_dir = split_dir / "train"
val_dir = split_dir / "val"
test_dir = split_dir / "test"
for d in (train_dir, val_dir, test_dir):
    d.mkdir(parents=True, exist_ok=True)

TRAIN_RATIO, VAL_RATIO = 0.8, 0.1
rng = random.Random(SEED)

for cls_dir in sorted(p for p in raw_dir.iterdir() if p.is_dir()):
    images = sorted(p for p in cls_dir.iterdir() if p.is_file())
    rng.shuffle(images)
    t_end = int(len(images) * TRAIN_RATIO)
    v_end = t_end + int(len(images) * VAL_RATIO)
    partitions = {
        train_dir: images[:t_end],
        val_dir: images[t_end:v_end],
        test_dir: images[v_end:],
    }
    for base, files in partitions.items():
        dst_cls = base / cls_dir.name
        dst_cls.mkdir(parents=True, exist_ok=True)
        for src_file in files:
            shutil.copy2(src_file, dst_cls / src_file.name)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE


def cast_to_uint8(img, label):
    return tf.cast(tf.clip_by_value(tf.round(img), 0.0, 255.0), tf.uint8), label


def preprocess_mobilenet(img, label):
    return (tf.cast(img, tf.float32) / 127.5) - 1.0, label


def augment_and_preprocess(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    upscaled = tf.image.resize(img, [IMAGE_SIZE[0] + 28, IMAGE_SIZE[1] + 28])
    img = tf.image.random_crop(upscaled, size=[IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    img = tf.image.random_brightness(img, max_delta=25.0)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return (img / 127.5) - 1.0, label


def load_dir_ds(directory: Path, shuffle: bool):
    return tf.keras.utils.image_dataset_from_directory(
        str(directory),
        labels="inferred",
        label_mode="categorical",
        image_size=IMAGE_SIZE,
        batch_size=None,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
        crop_to_aspect_ratio=True,
    )


raw_train = load_dir_ds(train_dir, shuffle=True)
ordered_labels = list(raw_train.class_names)
raw_train_eval = load_dir_ds(train_dir, shuffle=False)
raw_val = load_dir_ds(val_dir, shuffle=False)
raw_test = load_dir_ds(test_dir, shuffle=False)

# Cache resized uint8 tensors in RAM (~1.8 GB total) so epochs 2..20 avoid disk/JPEG bottlenecks.
train_ds = (
    raw_train.map(cast_to_uint8, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(buffer_size=2048, seed=SEED, reshuffle_each_iteration=True)
    .map(augment_and_preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
train_eval_ds = (
    raw_train_eval.map(preprocess_mobilenet, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = (
    raw_val.map(cast_to_uint8, num_parallel_calls=AUTOTUNE)
    .cache()
    .map(preprocess_mobilenet, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    raw_test.map(preprocess_mobilenet, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Urutan label:", ordered_labels)

## 4. Arsitektur Model & 2-Stage Transfer Learning

In [ ]:
EMBEDDING_LAYER_NAME = "embedding_gap"
LABEL_SMOOTHING = 0.1
FINE_TUNE_LAYERS = 40
CKPT_PATH = "/content/best_fastfood_v2.weights.h5"


class ThresholdCallback(Callback):
    def __init__(self, target_acc=0.96):
        super().__init__()
        self.target_acc = target_acc

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if logs.get("accuracy", 0.0) >= self.target_acc and logs.get("val_accuracy", 0.0) >= self.target_acc:
            print(f"\nTarget >= {self.target_acc*100:.1f}% tercapai pada epoch {epoch+1}. Menghentikan training.")
            self.model.stop_training = True


base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.Conv2D(128, (3, 3), padding="same", use_bias=False, name="head_conv"),
    layers.BatchNormalization(name="head_bn"),
    layers.ReLU(name="head_relu"),
    layers.MaxPooling2D((2, 2), name="head_pool"),
    layers.GlobalAveragePooling2D(name=EMBEDDING_LAYER_NAME),
    layers.Dense(256, activation="relu", name="head_dense"),
    layers.Dropout(0.4, name="head_dropout"),
    layers.Dense(len(ordered_labels), activation="softmax", name="predictions"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)
model.summary()

print("\n--- Stage 1: Head Warmup (MobileNetV2 Frozen) ---")
hist1 = model.fit(
    train_ds,
    epochs=8,
    validation_data=val_ds,
    callbacks=[
        ModelCheckpoint(CKPT_PATH, monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ],
    verbose=1,
)

print(f"\n--- Stage 2: Fine-Tuning {FINE_TUNE_LAYERS} Layer Atas MobileNetV2 ---")
base_model.trainable = True
freeze_until = max(0, len(base_model.layers) - FINE_TUNE_LAYERS)
for idx, layer in enumerate(base_model.layers):
    if idx < freeze_until or isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

hist2 = model.fit(
    train_ds,
    epochs=12,
    validation_data=val_ds,
    callbacks=[
        ModelCheckpoint(CKPT_PATH, monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1),
        ThresholdCallback(target_acc=0.96),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1),
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ],
    verbose=1,
)

if os.path.exists(CKPT_PATH):
    model.load_weights(CKPT_PATH)
    print("Bobot checkpoint terbaik dimuat kembali.")

## 5. Evaluasi Test Set, Classification Report, & Kurva Pelatihan

In [ ]:
def extract_targets(ds: tf.data.Dataset) -> np.ndarray:
    return np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in ds], axis=0)


test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

y_probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_probs, axis=1)
y_true = extract_targets(test_ds)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=ordered_labels, digits=4))

acc = hist1.history["accuracy"] + hist2.history["accuracy"]
val_acc = hist1.history["val_accuracy"] + hist2.history["val_accuracy"]
loss = hist1.history["loss"] + hist2.history["loss"]
val_loss = hist1.history["val_loss"] + hist2.history["val_loss"]
stage1_len = len(hist1.history["accuracy"])
epochs_range = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_range, acc, label="Training Accuracy", linewidth=2)
axes[0].plot(epochs_range, val_acc, label="Validation Accuracy", linewidth=2)
axes[0].axvline(stage1_len + 0.5, color="gray", linestyle="--", label="Start Fine-Tuning")
axes[0].set_title("Training & Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(loc="lower right")

axes[1].plot(epochs_range, loss, label="Training Loss", linewidth=2)
axes[1].plot(epochs_range, val_loss, label="Validation Loss", linewidth=2)
axes[1].axvline(stage1_len + 0.5, color="gray", linestyle="--", label="Start Fine-Tuning")
axes[1].set_title("Training & Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 6. Kalibrasi OOD Embedding Gate (128-dim Centroids)

In [ ]:
def l2_normalize(vecs: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vecs, axis=-1, keepdims=True)
    return vecs / np.maximum(norms, 1e-12)


emb_layer = model.get_layer(EMBEDDING_LAYER_NAME)
extractor = tf.keras.Model(inputs=model.inputs, outputs=[emb_layer.output, model.output])

train_embs, _ = extractor.predict(train_eval_ds, verbose=0)
train_embs = l2_normalize(train_embs)
train_targets = extract_targets(train_eval_ds)

centroids = {}
for idx, label in enumerate(ordered_labels):
    if label == OTHER_CLASS_NAME:
        continue
    cls_embs = train_embs[train_targets == idx]
    centroid = l2_normalize(cls_embs.mean(axis=0, keepdims=True))[0]
    centroids[label] = [round(float(v), 6) for v in centroid]

val_embs, _ = extractor.predict(val_ds, verbose=0)
val_embs = l2_normalize(val_embs)
val_targets = extract_targets(val_ds)

in_dist_sims = []
other_sims = []
centroid_matrix = np.array([centroids[k] for k in centroids], dtype=np.float32)

for emb, target_idx in zip(val_embs, val_targets):
    label = ordered_labels[int(target_idx)]
    if label == OTHER_CLASS_NAME:
        other_sims.append(float(np.max(centroid_matrix @ emb)))
    else:
        c_vec = np.asarray(centroids[label], dtype=np.float32)
        in_dist_sims.append(float(np.dot(emb, c_vec)))

raw_threshold = float(np.percentile(in_dist_sims, 5.0))
cosine_threshold = round(float(np.clip(raw_threshold, 0.55, 0.82)), 4)

print(f"In-distribution val cosine sim median : {np.median(in_dist_sims):.4f}")
print(f"In-distribution val cosine sim 5th pct: {raw_threshold:.4f} -> threshold = {cosine_threshold}")
if other_sims:
    print(f"Other class val max cosine sim median : {np.median(other_sims):.4f}")

ood_config = {
    "enabled": True,
    "embedding_layer": EMBEDDING_LAYER_NAME,
    "min_cosine_similarity": cosine_threshold,
    "min_confidence": 0.60,
    "max_entropy": 1.15,
    "class_centroids": centroids,
}

## 7. Ekspor Multi-Format (`saved_model`, `tflite`, `tfjs_model`, & `model_meta.json`)

In [ ]:
export_dir = Path("/content/models_v2")
if export_dir.exists():
    shutil.rmtree(export_dir)

saved_model_dir = export_dir / "saved_model"
tflite_dir = export_dir / "tflite"
tfjs_dir = export_dir / "tfjs_model"
for d in (saved_model_dir, tflite_dir, tfjs_dir):
    d.mkdir(parents=True, exist_ok=True)

if hasattr(model, "export"):
    model.export(str(saved_model_dir))
else:
    tf.saved_model.save(model, str(saved_model_dir))
print("[OK] SavedModel:", saved_model_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()
(tflite_dir / "model.tflite").write_bytes(tflite_bytes)
(tflite_dir / "label.txt").write_text("\n".join(ordered_labels) + "\n", encoding="utf-8")
print("[OK] TFLite:", tflite_dir / "model.tflite")


def clean_initializer(obj):
    if isinstance(obj, dict) and "class_name" in obj and "config" in obj:
        return {"class_name": obj["class_name"], "config": obj["config"]}
    return obj


def clean_layer(layer):
    cname = layer.get("class_name")
    cfg = dict(layer.get("config", {}))
    if cname == "InputLayer":
        cfg.pop("optional", None)
    elif cname == "DepthwiseConv2D":
        cfg.pop("groups", None)
    elif cname == "Conv2D" and cfg.get("groups") == 1:
        cfg.pop("groups", None)
    for k in list(cfg.keys()):
        if k.endswith(("_initializer", "_regularizer", "_constraint")) and isinstance(cfg[k], dict):
            cfg[k] = clean_initializer(cfg[k])
    if cname in ("Functional", "Model", "Sequential") and "layers" in cfg:
        cfg["layers"] = [clean_layer(sub) for sub in cfg["layers"]]
    out = dict(layer)
    out["config"] = cfg
    return out


weight_data = bytearray()
weight_manifest = []
for w_tensor, var in zip(model.get_weights(), model.weights):
    weight_manifest.append({
        "name": var.name.split(":")[0],
        "shape": list(w_tensor.shape),
        "dtype": "float32",
    })
    weight_data.extend(w_tensor.astype(np.float32).tobytes())

bin_name = "group1-shard1of1.bin"
(tfjs_dir / bin_name).write_bytes(bytes(weight_data))

raw_topo = json.loads(model.to_json())
tfjs_manifest = {
    "format": "layers-model",
    "generatedBy": f"keras v{tf.__version__}",
    "convertedBy": "TensorFlow.js LayersModel Normalizer",
    "modelTopology": {
        "keras_version": raw_topo.get("keras_version", tf.__version__),
        "backend": raw_topo.get("backend", "tensorflow"),
        "model_config": clean_layer({"class_name": raw_topo["class_name"], "config": raw_topo["config"]}),
    },
    "weightsManifest": [{"paths": [bin_name], "weights": weight_manifest}],
}
(tfjs_dir / "model.json").write_text(json.dumps(tfjs_manifest, indent=2), encoding="utf-8")
print("[OK] TFJS:", tfjs_dir)

model_meta = {
    "version": "v2",
    "preprocessing": "mobilenet_v2",
    "crop": "center_square",
    "input_size": [224, 224],
    "labels": ordered_labels,
    "other_class": OTHER_CLASS_NAME if OTHER_CLASS_NAME in ordered_labels else None,
    "ood": ood_config,
}
(export_dir / "model_meta.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")
print("[OK] Metadata:", export_dir / "model_meta.json")

## 8. Uji Inferensi TFLite & Kemas `models_v2.zip` untuk Repo Lokal

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_dir / "model.tflite"))
interpreter.allocate_tensors()
in_det = interpreter.get_input_details()
out_det = interpreter.get_output_details()


def predict_tflite(img_path: Path):
    with Image.open(img_path) as raw:
        im = raw.convert("RGB")
        w, h = im.size
        side = min(w, h)
        im = im.crop(((w - side) // 2, (h - side) // 2, (w + side) // 2, (h + side) // 2))
        im_resized = im.resize((224, 224), Image.Resampling.BILINEAR)
        arr = (np.asarray(im_resized, dtype=np.float32) / 127.5) - 1.0
    interpreter.set_tensor(in_det[0]["index"], np.expand_dims(arr, axis=0))
    interpreter.invoke()
    probs = interpreter.get_tensor(out_det[0]["index"])[0]
    idx = int(np.argmax(probs))
    return ordered_labels[idx], float(probs[idx]) * 100.0, im_resized


samples = []
for cls_dir in sorted(p for p in test_dir.iterdir() if p.is_dir()):
    files = list(cls_dir.iterdir())
    if files:
        samples.append((cls_dir.name, random.choice(files)))

plt.figure(figsize=(18, 4))
for i, (true_cls, img_p) in enumerate(samples):
    pred_lbl, conf, disp = predict_tflite(img_p)
    plt.subplot(1, len(samples), i + 1)
    plt.imshow(disp)
    color = "green" if pred_lbl == true_cls else "red"
    plt.title(f"Pred: {pred_lbl}\n({conf:.1f}%)\nTrue: {true_cls}", color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

zip_path = Path("/content/models_v2.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive("/content/models_v2", "zip", root_dir=str(export_dir))
print(f"\n[SUKSES] Arsip siap diunduh: {zip_path} ({zip_path.stat().st_size / (1024*1024):.1f} MB)")
print("Ekstrak isi models_v2.zip ke folder fast-food-image-classification/models/ lalu muat ulang web demo.")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass